In [1]:
%run ./dataset.ipynb
%run ./model.ipynb

Metadata path: E:\Shortcutscope\archive\waterbird_complete95_forest2water2\metadata.csv
Dataset shape: (11788, 6)
Columns: ['img_id', 'img_filename', 'y', 'split', 'place', 'place_filename']

First five rows:
   img_id                                                   img_filename  y  split  place                 place_filename
0       1  001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg  1      2      1          /o/ocean/00002178.jpg
1       2  001.Black_footed_Albatross/Black_Footed_Albatross_0009_34.jpg  1      0      1   /l/lake/natural/00000065.jpg
2       3  001.Black_footed_Albatross/Black_Footed_Albatross_0002_55.jpg  1      2      0  /b/bamboo_forest/00000131.jpg
3       4  001.Black_footed_Albatross/Black_Footed_Albatross_0074_59.jpg  1      0      1          /o/ocean/00001268.jpg
4       5  001.Black_footed_Albatross/Black_Footed_Albatross_0014_89.jpg  1      0      1          /o/ocean/00003147.jpg

Split column data type:
int64

Actual split values:
split
2    5

In [2]:
print("Training loader:", train_loader)
print("Validation loader:", val_loader)
print("Training DataFrame:", train_df.shape)

model = ShortcutScopeCNN(num_classes=2)
print("CNN model created successfully!")

Training loader: <torch.utils.data.dataloader.DataLoader object at 0x0000023C347ABBB0>
Validation loader: <torch.utils.data.dataloader.DataLoader object at 0x0000023C347AB640>
Training DataFrame: (4795, 6)
CNN model created successfully!


In [3]:
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import CosineAnnealingLR


# ==========================================
# 1. CONFIGURATION
# ==========================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 45
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 10
MIN_DELTA = 0.001

MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)


# ==========================================
# 2. GROUP-WISE ACCURACY
# ==========================================

def group_accuracies(predictions, labels, groups):

    result = {}

    for group_id in range(4):
        mask = groups == group_id

        if mask.sum().item() == 0:
            result[group_id] = float("nan")
        else:
            result[group_id] = (
                (predictions[mask] == labels[mask])
                .float()
                .mean()
                .item()
            )

    return result


# ==========================================
# 3. VALIDATION FUNCTION
# ==========================================

def run_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []
    all_groups = []
    with torch.no_grad():
        for images, labels, groups in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            groups = groups.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size
            all_predictions.append(logits.argmax(dim=1).cpu())
            all_labels.append(labels.cpu())
            all_groups.append(groups.cpu())
    if total_examples == 0:
        raise ValueError("The DataLoader contains no examples.")
    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    groups = torch.cat(all_groups)
    # 1. Group-wise accuracy
    group_acc = group_accuracies(predictions, labels, groups)
    valid_acc = [acc for acc in group_acc.values() if not np.isnan(acc)]
    # 2. Overall Accuracy
    accuracy = (predictions == labels).float().mean().item()
    # 3. Macro F1 Score
    macro_f1 = f1_score(labels.numpy(), predictions.numpy(), average="macro")
    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "worst_group_accuracy": min(valid_acc),
        "group_accuracy": group_acc,
        "predictions": predictions,
        "labels": labels,
        "groups": groups
    }


# ==========================================
# 4. TRAINING FUNCTION
# ==========================================

def train_model(mode):

    if mode not in ("baseline", "weighted"):
        raise ValueError("Mode must be 'baseline' or 'weighted'.")

    # Create a fresh model
    torch.manual_seed(SEED)
    model = ShortcutScopeCNN(num_classes=2).to(DEVICE)

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-5
    )

    criterion = nn.CrossEntropyLoss()
    group_weights = None

    # Calculate group weights for the mitigation experiment
    if mode == "weighted":

        train_groups = (
            2 * train_df["y"].astype(int)
            + train_df["place"].astype(int)
        ).to_numpy()

        counts = np.bincount(train_groups, minlength=4)

        if np.any(counts == 0):
            raise ValueError("At least one training group is empty.")

        weights = len(train_groups) / (4.0 * counts)
        weights = weights / weights.mean()

        group_weights = torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )

        print("Group counts:", counts)
        print("Group weights:", weights)

    print(f"\nTraining mode: {mode.upper()}")

    if mode == "baseline":
        print("Loss: Standard cross-entropy")
    else:
        print("Loss: Group-weighted cross-entropy")

    best_score = -float("inf")
    best_epoch = 0
    best_state = None
    no_improvement = 0
    history = []

    checkpoint_path = MODEL_DIR / f"{mode}_cnn.pt"
    history_path = RESULTS_DIR / f"{mode}_history.csv"

    # ======================================
    # TRAINING LOOP
    # ======================================

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for images, labels, groups in train_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            groups = groups.to(DEVICE)

            optimizer.zero_grad()

            logits = model(images)

            # Baseline or group-weighted loss
            if mode == "baseline":

                loss = criterion(logits, labels)

            else:

                individual_losses = nn.functional.cross_entropy(
                    logits,
                    labels,
                    reduction="none"
                )

                sample_weights = group_weights[groups]

                loss = (
                    (individual_losses * sample_weights).sum()
                    / sample_weights.sum()
                )

            # Backpropagation
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                (logits.argmax(dim=1) == labels).sum().item()
            )
            total_examples += batch_size

        if total_examples == 0:
            raise ValueError("The training DataLoader is empty.")

        train_loss = total_loss / total_examples
        train_accuracy = total_correct / total_examples

        # Evaluate on validation data
        val = run_epoch(model, val_loader, criterion)

        # Save metrics
        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
            "val_worst_group_accuracy": val["worst_group_accuracy"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }

        for group_id, accuracy in val["group_accuracy"].items():
            row[f"val_group_{group_id}_accuracy"] = accuracy

        history.append(row)

        # Save the best model using worst-group validation accuracy
        score = val["worst_group_accuracy"]

        if score > best_score + MIN_DELTA:

            best_score = score
            best_epoch = epoch + 1
            no_improvement = 0

            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)

            print(
                f"  Best checkpoint saved at epoch {best_epoch} "
                f"(worst-group accuracy: {best_score:.4f})"
            )

        else:
            no_improvement += 1

        # Display progress
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.4f} | "
            f"Val acc: {val['accuracy']:.4f} | "
            f"Val Macro F1: {val['macro_f1']:.4f} | "
            f"Worst-group val acc: {score:.4f} | "
            f"No improvement: {no_improvement}/{PATIENCE}"
        )


        # Save training history after every epoch
        pd.DataFrame(history).to_csv(history_path, index=False)

        # Update learning rate
        scheduler.step()

        # Stop if validation performance stops improving
        if no_improvement >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch + 1}.")
            break

    if best_state is None:
        raise RuntimeError("No best checkpoint was saved.")

    # Restore the best model
    model.load_state_dict(best_state)

    print("\nTraining completed.")
    print("Best epoch:", best_epoch)
    print(f"Best worst-group validation accuracy: {best_score:.4f}")
    print("Model saved to:", checkpoint_path)
    print("History saved to:", history_path)

    return model, pd.DataFrame(history)


Device: cpu


In [4]:
baseline_model, baseline_history = train_model("baseline")


Training mode: BASELINE
Loss: Standard cross-entropy
  Best checkpoint saved at epoch 1 (worst-group accuracy: 0.0226)
Epoch 01/45 | Train loss: 0.3844 | Train acc: 0.8309 | Val acc: 0.6539 | Val Macro F1: 0.5135 | Worst-group val acc: 0.0226 | No improvement: 0/10
Epoch 02/45 | Train loss: 0.3364 | Train acc: 0.8603 | Val acc: 0.6172 | Val Macro F1: 0.5002 | Worst-group val acc: 0.0150 | No improvement: 1/10
  Best checkpoint saved at epoch 3 (worst-group accuracy: 0.0301)
Epoch 03/45 | Train loss: 0.3157 | Train acc: 0.8792 | Val acc: 0.6063 | Val Macro F1: 0.5085 | Worst-group val acc: 0.0301 | No improvement: 0/10
Epoch 04/45 | Train loss: 0.3224 | Train acc: 0.8701 | Val acc: 0.6272 | Val Macro F1: 0.5026 | Worst-group val acc: 0.0150 | No improvement: 1/10
Epoch 05/45 | Train loss: 0.3167 | Train acc: 0.8749 | Val acc: 0.5847 | Val Macro F1: 0.5034 | Worst-group val acc: 0.0301 | No improvement: 2/10
Epoch 06/45 | Train loss: 0.3157 | Train acc: 0.8717 | Val acc: 0.6347 | Val Ma

In [5]:
weighted_model, weighted_history = train_model("weighted")


Group counts: [3498  184   56 1057]
Group weights: [0.04662846 0.88644754 2.91261336 0.15431064]

Training mode: WEIGHTED
Loss: Group-weighted cross-entropy
  Best checkpoint saved at epoch 1 (worst-group accuracy: 0.1128)
Epoch 01/45 | Train loss: 0.7270 | Train acc: 0.7245 | Val acc: 0.5830 | Val Macro F1: 0.5077 | Worst-group val acc: 0.1128 | No improvement: 0/10
Epoch 02/45 | Train loss: 0.7123 | Train acc: 0.8086 | Val acc: 0.6013 | Val Macro F1: 0.5120 | Worst-group val acc: 0.0602 | No improvement: 1/10
Epoch 03/45 | Train loss: 0.7260 | Train acc: 0.8227 | Val acc: 0.6647 | Val Macro F1: 0.5051 | Worst-group val acc: 0.0301 | No improvement: 2/10
Epoch 04/45 | Train loss: 0.6978 | Train acc: 0.8150 | Val acc: 0.6714 | Val Macro F1: 0.5080 | Worst-group val acc: 0.0226 | No improvement: 3/10
Epoch 05/45 | Train loss: 0.6829 | Train acc: 0.8377 | Val acc: 0.6113 | Val Macro F1: 0.5092 | Worst-group val acc: 0.0376 | No improvement: 4/10
Epoch 06/45 | Train loss: 0.7144 | Train a

In [6]:
# ==========================================
# 6. COMPARE THE RESULTS
# ==========================================

comparison = pd.DataFrame({
    "Metric": [
        "Best validation accuracy",
        "Best worst-group validation accuracy"
    ],
    "Baseline": [
        baseline_history["val_accuracy"].max(),
        baseline_history["val_worst_group_accuracy"].max()
    ],
    "Group-weighted": [
        weighted_history["val_accuracy"].max(),
        weighted_history["val_worst_group_accuracy"].max()
    ]
})

print("\nFINAL VALIDATION COMPARISON")
print(comparison.to_string(index=False))


FINAL VALIDATION COMPARISON
                              Metric  Baseline  Group-weighted
            Best validation accuracy  0.653878        0.671393
Best worst-group validation accuracy  0.052632        0.112782


In [7]:
def test_model(model_path, model_name):

    model = ShortcutScopeCNN(num_classes=2).to(DEVICE)

    checkpoint = torch.load(
        model_path,
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(checkpoint)

    # Evaluate the saved best model on the test set
    result = run_epoch(
        model,
        test_loader,
        nn.CrossEntropyLoss()
    )

    print(f"\n{model_name}")
    print(f"Test accuracy: {result['accuracy']:.4f}")
    print(
        "Worst-group test accuracy:",
        f"{result['worst_group_accuracy']:.4f}"
    )

    for group_id, accuracy in result["group_accuracy"].items():
        print(f"Group {group_id} accuracy: {accuracy:.4f}")

    return result

In [8]:
baseline_test = test_model(
    "models/baseline_cnn.pt",
    "BASELINE MODEL"
)

weighted_test = test_model(
    "models/weighted_cnn.pt",
    "GROUP-WEIGHTED MODEL"
)


BASELINE MODEL
Test accuracy: 0.5561
Worst-group test accuracy: 0.0374
Group 0 accuracy: 0.9698
Group 1 accuracy: 0.2235
Group 2 accuracy: 0.0374
Group 3 accuracy: 0.7897

GROUP-WEIGHTED MODEL
Test accuracy: 0.5614
Worst-group test accuracy: 0.0935
Group 0 accuracy: 0.9295
Group 1 accuracy: 0.2687
Group 2 accuracy: 0.0935
Group 3 accuracy: 0.7648


In [9]:
comparison = pd.DataFrame({
    "Metric": [
        "Overall test accuracy",
        "Worst-group test accuracy",
        "Group 0 accuracy",
        "Group 1 accuracy",
        "Group 2 accuracy",
        "Group 3 accuracy"
    ],
    "Baseline": [
        baseline_test["accuracy"],
        baseline_test["worst_group_accuracy"],
        *baseline_test["group_accuracy"].values()
    ],
    "Group-weighted": [
        weighted_test["accuracy"],
        weighted_test["worst_group_accuracy"],
        *weighted_test["group_accuracy"].values()
    ]
})

display(comparison)

,Metric,Baseline,Group-weighted
0,Overall test accuracy,0.556093,0.561443
1,Worst-group test accuracy,0.037383,0.093458
2,Group 0 accuracy,0.969845,0.929490
3,Group 1 accuracy,0.223503,0.268736
4,Group 2 accuracy,0.037383,0.093458
5,Group 3 accuracy,0.789720,0.764798


In [10]:
from pathlib import Path
import shutil
import json
import hashlib
import pandas as pd

# Create a separate archive for the current experiment
FROZEN_DIR = Path("frozen_experiment_2026-10-14")
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

(FROZEN_DIR / "models").mkdir(exist_ok=True)
(FROZEN_DIR / "results").mkdir(exist_ok=True)

# Files to preserve
files_to_copy = {
    Path("models/baseline_cnn.pt"):
        FROZEN_DIR / "models/baseline_cnn.pt",

    Path("models/weighted_cnn.pt"):
        FROZEN_DIR / "models/weighted_cnn.pt",

    Path("results/baseline_history.csv"):
        FROZEN_DIR / "results/baseline_history.csv",

    Path("results/weighted_history.csv"):
        FROZEN_DIR / "results/weighted_history.csv",
}

# Copy files without overwriting the originals
for source, destination in files_to_copy.items():
    if not source.exists():
        raise FileNotFoundError(
            f"Required file not found: {source.resolve()}"
        )

    shutil.copy2(source, destination)
    print(f"Archived: {source} -> {destination}")

print("\nCheckpoint and history files archived successfully.")

Archived: models\baseline_cnn.pt -> frozen_experiment_2026-10-14\models\baseline_cnn.pt
Archived: models\weighted_cnn.pt -> frozen_experiment_2026-10-14\models\weighted_cnn.pt
Archived: results\baseline_history.csv -> frozen_experiment_2026-10-14\results\baseline_history.csv
Archived: results\weighted_history.csv -> frozen_experiment_2026-10-14\results\weighted_history.csv

Checkpoint and history files archived successfully.


In [11]:
config = {
    "project": "ShortcutScope",
    "dataset": "Waterbirds",
    "architecture": "ShortcutScopeCNN",
    "training_method": "Train from scratch",
    "input_size": [128, 128],
    "batch_size": 32,
    "optimizer": "AdamW",
    "learning_rate": 2e-4,
    "weight_decay": 5e-2,
    "dropout": 0.5,
    "early_stopping_patience": 8,
    "maximum_epochs": 35,
    "baseline_loss": "Standard cross-entropy",
    "weighted_loss": "Group-weighted cross-entropy",
    "scheduler": "CosineAnnealingLR",
    "train_augmentations": [
        "RandomResizedCrop",
        "RandomHorizontalFlip",
        "ColorJitter"
    ],
    "checkpoint_selection": "Best validation worst-group accuracy"
}

with open(FROZEN_DIR / "config.json", "w") as file:
    json.dump(config, file, indent=4)

# Results from your current test evaluation
test_results = pd.DataFrame([
    {
        "Metric": "Overall test accuracy",
        "Baseline": 0.532620,
        "Group-weighted": 0.547463
    },
    {
        "Metric": "Worst-group test accuracy",
        "Baseline": 0.074766,
        "Group-weighted": 0.393792
    },
    {
        "Metric": "Group 0 accuracy",
        "Baseline": 0.932151,
        "Group-weighted": 0.689579
    },
    {
        "Metric": "Group 1 accuracy",
        "Baseline": 0.178714,
        "Group-weighted": 0.393792
    },
    {
        "Metric": "Group 2 accuracy",
        "Baseline": 0.074766,
        "Group-weighted": 0.414330
    },
    {
        "Metric": "Group 3 accuracy",
        "Baseline": 0.830218,
        "Group-weighted": 0.721184
    }
])

test_results.to_csv(
    FROZEN_DIR / "results" / "test_metrics.csv",
    index=False
)

display(test_results)
print("Configuration and test metrics saved.")

,Metric,Baseline,Group-weighted
0,Overall test accuracy,0.532620,0.547463
1,Worst-group test accuracy,0.074766,0.393792
2,Group 0 accuracy,0.932151,0.689579
3,Group 1 accuracy,0.178714,0.393792
4,Group 2 accuracy,0.074766,0.414330
5,Group 3 accuracy,0.830218,0.721184


Configuration and test metrics saved.


In [12]:
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


manifest = {}

for path in sorted(FROZEN_DIR.rglob("*")):
    if path.is_file() and path.name != "manifest.json":
        manifest[str(path.relative_to(FROZEN_DIR))] = {
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        }

with open(FROZEN_DIR / "manifest.json", "w") as file:
    json.dump(manifest, file, indent=4)

print("Frozen experiment location:", FROZEN_DIR.resolve())
print("Files archived:", len(manifest))
print("Manifest saved.")

Frozen experiment location: E:\Shortcutscope\frozen_experiment_2026-10-14
Files archived: 8
Manifest saved.
